# 05 Manual check of unlabelled pairs

Quora labels only a small share of the pairs, so `precision_implied` counts every unlabelled pair as wrong
(Union + Louvain: 0.007). Here 100 unlabelled pairs of the final system are labelled by hand to estimate the
real precision of the two things the project adds on top of LSH:

- **embedding only**: cosine ≥ the final threshold, never returned by LSH (at any score);
- **Louvain transitive**: same Louvain cluster, but not an edge of the final graph.

*Unlabelled* = no Quora label (neither 1 nor 0) and not in the same true duplicate group.
Only saved outputs of 02–04 are used. Louvain labels are not saved by 04, so they are rebuilt with saved
setting and seed.

In [ ]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd

from src.manual_check import (rebuild_final_clusters, embedding_only_pool, louvain_transitive_sample,
                              build_sheet, export_sheet, load_labels, summarise_labels)

D = "../data/processed/"
SHEET_PATH = D + "manual_check.csv" # pairs to label
KEY_PATH = D + "manual_check_key.csv" # id -> method and qids
SEED = 42
N_PER_METHOD = 50

questions = pd.read_parquet(D + "questions.parquet")
pairs = pd.read_parquet(D + "pairs.parquet")
N = len(questions)
assert (questions["qid"].to_numpy() == np.arange(N)).all()   # qids are 0..N-1, used as array indexes
lsh_all = pd.read_parquet(D + "lsh_candidates.parquet", columns=["qid_a", "qid_b", "score"])
emb = pd.read_parquet(D + "emb_candidates.parquet", columns=["qid_a", "qid_b", "score"])

## 1. The final system of clustering and the two pools

In [ ]:
setting, edges, labels = rebuild_final_clusters(lsh_all, emb, N, D + "p4_sweep_dev.csv")
print(f"Clustering setting: Jaccard >= {setting['lsh_thr']}, cosine >= {setting['emb_thr']}, gamma = {setting['gamma']:g} "
      f"| {len(edges):,} edges")

emb_pool = embedding_only_pool(emb, lsh_all, pairs, questions, setting["emb_thr"])
emb_sample = emb_pool.sample(N_PER_METHOD, random_state=SEED)
louvain_sample, share_kept = louvain_transitive_sample(labels, edges, pairs, questions, N_PER_METHOD, SEED)
print(f"embedding-only pool: {len(emb_pool):,} unlabelled pairs")
print(f"Louvain transitive : {share_kept:.1%} of the pairs inside clusters are unlabelled and not an edge")

P4 setting: Jaccard >= 0.9, cosine >= 0.75, gamma = 20 | 1,213,339 edges
embedding-only pool: 811,944 unlabelled pairs
Louvain transitive : 94.7% of the pairs inside clusters are unlabelled and not an edge


## 2. Sheet to label

In [3]:
sheet = build_sheet({"embedding only": emb_sample, "Louvain transitive": louvain_sample}, questions, SEED)
export_sheet(sheet, SHEET_PATH, KEY_PATH)

wrote manual_check.csv (100 pairs to label) and manual_check_key.csv


**Labelling rule.** Fill the `label` column of `manual_check.csv`:

- `1` if **one answer satisfies both questions**;
- `0` if not (related topic is not enough: different person, place, product, year or direction).

The method of each pair is in `manual_check_key.csv`; do not open it while labelling.

## 3. Real precision per method

In [6]:
labelled = load_labels(SHEET_PATH, KEY_PATH)
if labelled is None:
    print("No labels yet: fill the label column of manual_check.csv and rerun this cell.")
else:
    display(summarise_labels(labelled).round(3))

,n_decided,n_unsure,n_not_labelled,share_duplicates,ci95_low,ci95_high
method,,,,,,
Louvain transitive,50,0,0,0.00,0.000,0.071
embedding only,50,0,0,0.52,0.385,0.652


Share of true duplicates among the labelled pairs, with a Wilson 95% interval.

In [7]:
if labelled is not None:
    pd.set_option("display.max_colwidth", 120)
    examples = labelled[labelled["label"].isin(["1", "0"])].groupby(["method", "label"]).head(1)
    display(examples[["method", "label", "q1", "q2"]].sort_values(["method", "label"]))

,method,label,q1,q2
0,Louvain transitive,0,how can i improve my scientific communication skills?,can someone correct my grammar and improve the sentence?
3,embedding only,0,which is best laptop below 40k?,which laptop or notebook can i buy under 20k?
9,embedding only,1,what are some projects in java suitable for beginners?,what is a good way to learn java for beginners?


## Conclusion

About half of the embedding only pairs are real duplicates (26 of 50, 52%, 95% CI 38% to 65%). Quora gives these pairs no label and LSH never returns them. So the low implied precision of the embedding pairs (0.061 in notebook 04) comes mostly from missing labels, not from wrong pairs.

None of the 50 Louvain transitive pairs is a real duplicate (95% CI 0% to 7%). These pairs are 94.7% of all the pairs inside the Louvain clusters. So the low implied precision of Union + Louvain (0.007) is real. Louvain joins unrelated questions, and transitivity cannot be trusted at this setting.

Both estimates are conservative, because one person labelled all pairs with a strict rule. The Louvain sample is uniform over pairs, so the large clusters dominate it. Small clusters may be more precise.